# 04 — Validate, Export, and Push Weights to GitHub (Equities)
Validates, copies, and version-tags trained checkpoint(s) from Google Drive
into `algo_trader/weights/`, then commits and pushes them.

For the current project state, expected artifacts are equities symbol files such as:
- `AAPL.pth`
- `MSFT.pth`
- `NVDA.pth`

**Checkpoint format** (saved by `DeepScalperAgent.save()`):
```
{
  "online_net":   state_dict,
  "target_net":   state_dict,
  "optimizer":    state_dict,
  "explore_rate": float,
  "epsilon":      float,
  "steps":        int,
}
```
`strategy.py` and validation code support both this wrapped format and a raw `state_dict` (legacy).

**Also committed (optional):** `training_log.csv` produced by `03_train_deepscalper.ipynb`.

**Prerequisites:**
- `GITHUB_TOKEN` in Colab Secrets with `repo` scope.
- `GITHUB_USERNAME` and `GITHUB_REPO` set in the config cell below.

Warning: this notebook pushes directly to `main`.
For production use, push to a branch and open a PR.

In [ ]:
!pip install -q torch gitpython

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

# ─── EDIT THESE ──────────────────────────────────────────────────────────────
GITHUB_USERNAME = 'YOUR_GITHUB_USERNAME'
GITHUB_REPO     = 'deepscalper_copilot'
# ─────────────────────────────────────────────────────────────────────────────

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise RuntimeError('Add GITHUB_TOKEN (with repo scope) to Colab Secrets.')

import os, shutil, sys

DRIVE_WEIGHTS  = '/content/drive/MyDrive/algo_trader/weights'
DRIVE_LOG      = '/content/drive/MyDrive/algo_trader/training_log.csv'
REPO_DIR       = '/content/deepscalper_copilot'
REPO_WEIGHTS   = f'{REPO_DIR}/algo_trader/weights'

# Make the repo importable so we can validate checkpoints with DeepScalperNet
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
ALGO_DIR = REPO_DIR + '/algo_trader'
if ALGO_DIR not in sys.path:
    sys.path.insert(0, ALGO_DIR)

print('Paths configured ✓')

In [ ]:
import git

REMOTE_URL = (
    f'https://{GITHUB_USERNAME}:{GITHUB_TOKEN}'
    f'@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'
)

if os.path.exists(REPO_DIR + '/.git'):
    repo = git.Repo(REPO_DIR)
    print('Repo already cloned — pulling latest changes...')
    origin = repo.remote('origin')
    origin.set_url(REMOTE_URL)
    origin.pull('main')
elif os.path.exists(REPO_DIR):
    print('Found leftover non-git directory — removing it before clone...')
    shutil.rmtree(REPO_DIR)
    repo = git.Repo.clone_from(REMOTE_URL, REPO_DIR)
else:
    print('Cloning repo...')
    repo = git.Repo.clone_from(REMOTE_URL, REPO_DIR)

print(f'HEAD: {repo.head.commit.hexsha[:8]}  ({repo.head.commit.message.strip()})')

In [ ]:
import importlib.util
import shutil
import subprocess
import sys
import torch
from pathlib import Path

repo_root = globals().get('REPO_DIR', '/content/deepscalper_copilot')
algo_root = globals().get('ALGO_DIR', f'{repo_root}/algo_trader')
drive_weights = globals().get('DRIVE_WEIGHTS', '/content/drive/MyDrive/algo_trader/weights')
repo_weights = globals().get('REPO_WEIGHTS', f'{algo_root}/weights')
repo_url = globals().get('REPO_URL')

if not repo_url:
    github_username = globals().get('GITHUB_USERNAME')
    github_repo = globals().get('GITHUB_REPO', 'deepscalper_copilot')
    if github_username and github_username != 'YOUR_GITHUB_USERNAME':
        repo_url = f'https://github.com/{github_username}/{github_repo}.git'
    else:
        repo_url = 'https://github.com/rohanpatrick568/deepscalper_copilot.git'

architecture_py = Path(repo_root) / 'algo_trader' / 'colab' / 'deepscalper' / 'architecture.py'


def _ensure_repo_checkout() -> None:
    if architecture_py.exists():
        return

    repo_path = Path(repo_root)
    if repo_path.exists():
        print('Repo checkout is missing DeepScalper sources - refreshing clone...')
        shutil.rmtree(repo_path)

    print(f'Bootstrapping repo checkout from {repo_url} ...')
    subprocess.run(['git', 'clone', '--depth', '1', repo_url, repo_root], check=True)

    if not architecture_py.exists():
        raise RuntimeError(
            f'DeepScalper architecture not found after clone: {architecture_py}. '
            'Check GITHUB_USERNAME/GITHUB_REPO or use the default project repo.'
        )


_ensure_repo_checkout()

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
if algo_root not in sys.path:
    sys.path.insert(0, algo_root)

spec = importlib.util.spec_from_file_location('deepscalper_architecture', architecture_py)
if spec is None or spec.loader is None:
    raise RuntimeError(f'Failed to load module spec for {architecture_py}')
architecture_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(architecture_module)
DeepScalperNet = architecture_module.DeepScalperNet

os.makedirs(repo_weights, exist_ok=True)

pth_files = sorted(Path(drive_weights).glob('*.pth'))
print(f'Found {len(pth_files)} weight file(s) in Drive.\n')

copied = []
skipped = []

for src in pth_files:
    # Validate checkpoint format against the current DeepScalperNet definition.
    try:
        ckpt = torch.load(str(src), map_location='cpu', weights_only=True)
    except Exception as error:
        print(f'  SKIP {src.name}: could not load - {error}')
        skipped.append(src.name)
        continue

    state_dict = ckpt.get('online_net', ckpt)

    try:
        net = DeepScalperNet()
        net.load_state_dict(state_dict, strict=True)
    except Exception as error:
        print(f'  SKIP {src.name}: incompatible with DeepScalperNet - {error}')
        skipped.append(src.name)
        continue

    dst = Path(repo_weights) / src.name
    shutil.copy2(src, dst)
    copied.append(str(dst))
    print(f'  OK   {src.name}')

print(f'\nCopied {len(copied)} / {len(pth_files)} files -> {repo_weights}')
if skipped:
    print(f'Skipped: {skipped}')

In [ ]:
from datetime import datetime, timezone

if not copied:
    raise RuntimeError('No valid weight files to commit. Check the validation output above.')

# Stage .pth weight files
staged_paths = [str(Path(f).relative_to(REPO_DIR)) for f in copied]
repo.index.add(staged_paths)

# Stage training_log.csv if present on Drive
log_dst = Path(REPO_DIR) / 'algo_trader' / 'training_log.csv'
if os.path.exists(DRIVE_LOG):
    shutil.copy2(DRIVE_LOG, log_dst)
    repo.index.add([str(log_dst.relative_to(REPO_DIR))])
    print(f'Staged training_log.csv ({log_dst})')

if repo.is_dirty():
    timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%d-%H%M')
    tag_name = f'equities-weights-{timestamp}'
    commit_msg = (
        f'chore: export equities DeepScalper weights [{timestamp}] '
        f'({len(copied)} file(s))'
    )

    with repo.config_writer() as cfg:
        cfg.set_value('user', 'name',  GITHUB_USERNAME)
        cfg.set_value('user', 'email', f'{GITHUB_USERNAME}@users.noreply.github.com')

    commit = repo.index.commit(commit_msg)
    print(f'Committed: {commit.hexsha[:8]} - {commit_msg}')

    repo.create_tag(tag_name, ref=commit)
    print(f'Tag created: {tag_name}')

    origin = repo.remote('origin')
    origin.push('main')
    origin.push(tag_name)
    print(f'\nPushed to github.com/{GITHUB_USERNAME}/{GITHUB_REPO}  (tag: {tag_name})')
else:
    print('Nothing new to commit - repo is already up to date.')